# Project 07: Burrows-Wheeler Transform

**Note**: Keep the cell below as is. It is for formatting purposes

In [20]:
%%html
<style>
.MathJax_Display,  /* MathJax 2 */
mjx-container {    /* MathJax 3 */
  text-align: left !important;
}
</style>

BWT Algorithm

Phase A: Build the BWT
Step 1: Add the end marker. banana becomes banana$. The $ has ASCII value 36, which is smaller than every letter (A = 65, a = 97). So the row starting with $ always sorts to the top, and it marks where the text ends
Step 2: Make every rotation. Take the first letter and move it to the end, 7 times (one per character)
Step 3: Sort the rotations alphabetically. Each rotation remembers where it started in the original text
Step 4: Read down the last column. This gives the BWT (shortcut:  suffix array method)

Phase B: Build the index
Step 5: Build the count table. For each letter, count how many letters (ignoring $) are alphabetically smaller
Step 6: Build the occurrence table. Walk along the BWT annb$aa one position at a time. At each position, record a running tally of how many of each letter you’ve seen so far, including the current position

Phase C: Backward search
Step 7: Start with all rows. Set lower = 0 and upper = (length of the text including $) - 1. Then process the pattern right to left
	Each step uses the same two formulas from the slides:
	new lower = count[letter] + occur[letter][lower − 1] + 1. If lower is 0, use 0 in place of the occur term
	new upper = count[letter] + occur[letter][upper]
	If at any step lower > upper, the range is empty. The pattern isn’t there, so return []
Step 8: Turn rows into positions. Look up the surviving rows in the suffix array
	

Phase D: Compress
Step 9: Run-length encode, written as letter then run length.
Step 10: Run-length decode. Do the reverse: read a letter, read its number, and write that letter that many times



Functions in project below                      Matches Steps above             What it does
BWT(string)                                     Steps 1 - 4                     Builds the BWT using rotations
suffix_array(string)                            Step 3                          returns the starting positions of the sorted rotations
BWT_from_suffix_array(text, suffix_positiosn)   Step 4 shortcut                 builds the BWT from the suffix array
cal_count(string)                               Step 5                          builds the count table
cal_occur(bwt_string)                           Step 6                          builds the occurence table
update_range(lower, upper, count, occur, a)     Step 7, one letter              applies the two formulas once and returns the new lower and upper
find_match(query, refernce)                     Steps 7-8, whole search         runs the full search and returns the positions
run_length_encode(bwt_string)                   Step 9                          encodes the Step 4 result
run_length_decode(encode)                       Step 10                         decodes it back





BWT Algorithm Generic Pseudocode

Build The Transform
	add $ to the end of the text
	write every rotation of the text
	sort the rotations alphabetically
	BWT = the last letter of each sorted rotation
	suffix array = the starting positions of the sorted rotations, in order
	
	BWT (shortcut) = for each position p in the suffix array:
                     if p is 0: take the last letter of the text
                     else:      take the letter at p − 1	

Build The Index (once per reference)
	count[x] = how many letters (not $) are smaller than x
	occur[x][i] = how many x's appear in BWT positions 0...i

Search (once per query)
	lower = 0, upper = last row
	for each letter in the query, right to left:
		lower = count[letter] + occur[letter][lower - 1] + 1	(use 0 if lower is 0)
		upper = count[letter] + occur[letter][upper]
		if lower > upper: return no matches
	return the suffix-array values for rows lower...upper

Compress
	encode: walk the BWT, write each letter and how many times it repeats in a row
	decode: read letter and number, then write the letter that many times




In [21]:
def BWT(string: str) -> str:
    """Function to calculate Burrows-Wheeler Transform for a given string.
    
    Computes the Burrows-Wheeler Transform by creating all rotations of the input
    string, sorting them lexicographically, and extracting the last column.
    
    Args:
        string: The input string to transform.
    
    Returns:
        The Burrows-Wheeler Transform of the input string.
        
    Examples:
        >>> BWT('googol')
        'lo$oogg'
        
        >>> BWT('banana$')
        'annb$aa'
    """
    
    #IF string doesn't end with $, add $ to end of string
    text = string
    if not text.endswith("$"):
        text = text + "$"

    #Build each rotation, cut text & move front letter to back
    rotations = []
    for i in range(len(text)):
        rotation = text[i:] + text[:i]
        rotations.append(rotation)

    #sort the list of rotations lexicographically
    rotations.sort()

    #BWT is the last character of sorted rotation
    result = ""
    for rotation in rotations:
        result = result + rotation[-1]

    return result

In [22]:
#Test cell for BWT()

print(BWT('googol'))
print(BWT('banana'))


lo$oogg
annb$aa


In [23]:
def suffix_array(string: str) -> list[int]:
    """Function to calculate suffix-array for a given string.
    
    Computes the suffix array by sorting all suffixes of the input string
    lexicographically and returning their starting positions.
    
    Args:
        string: The input string to process.
    
    Returns:
        A list of integers representing the starting positions of the
        lexicographically sorted suffixes.
        
    Examples:
        >>> suffix_array('googol')
        [6, 3, 0, 5, 2, 4, 1]
        
        >>> suffix_array('banana$')
        [6, 5, 3, 1, 0, 4, 2]
    """

    # Add $ to end of text if missing
    text = string
    if not text.endswith("$"):
        text = text + "$"

    # Pair each suffix with position where it starts
    pairs = []
    for i in range(len(text)):
        suffix = text[i:]
        pairs.append((suffix, i))

    # Sort the list of pairs
    pairs.sort()

    # Keep only starting positions, now in sorted- suffix order
    result = []
    for pair in pairs:
        result.append(pair[1])

    return result

In [24]:
#Test cell for suffix_array()

print(suffix_array('googol'))
print(suffix_array('banana'))

[6, 3, 0, 5, 2, 4, 1]
[6, 5, 3, 1, 0, 4, 2]


In [25]:
def BWT_from_suffix_array(
  text: str, 
  suffix_positions: list[int]
  ) -> str:
    """Function to calculate the Burrows-Wheeler Transform from a suffix array.
    
    Computes the Burrows-Wheeler Transform by using the suffix array to identify
    the character that precedes each suffix in the sorted order.
    
    Args:
        text: The input string to transform.
        suffix_positions: The suffix array for the input string, containing the
            starting positions of all suffixes in lexicographical order.
    
    Returns:
        The Burrows-Wheeler Transform of the input string.
        
    Examples:
        >>> BWT_from_suffix_array("banana$", [6, 5, 3, 1, 0, 4, 2])
        'annb$aa'
        
        >>> BWT_from_suffix_array("googol$", [6, 3, 0, 5, 2, 4, 1])
        'lo$oogg'
    """
    
    # Add $ to end of text if missing
    if not text.endswith("$"):
        text = text + "$"

    # Each BWT character is the one just before its suffix in the text.
    # The suffix at position 0 has nothing before it, so wrap around to the last character instead.
    result = ""

    for p in suffix_positions:
        if p == 0:
            result = result + text[-1]
        else:
            result = result + text[p - 1]
    
    return result

In [26]:
#Test cell for BWT_from_suffix_array()

print(BWT_from_suffix_array("banana$", [6, 5, 3, 1, 0, 4, 2]))

print(BWT_from_suffix_array("googol$", [6, 3, 0, 5, 2, 4, 1]))

annb$aa
lo$oogg


In [27]:
from collections import Counter

def cal_count(string: str) -> dict[str, int]:
    """Function to count characters lexicographically smaller than each character.
    
    For each character in the alphabet, calculates how many characters in the
    input string are lexicographically smaller than it.
    
    Args:
        string: The input string to analyze.
    
    Returns:
        A dictionary mapping each character to the count of characters
        lexicographically smaller than it.
    
    Examples:
        >>> cal_count('ATGACG')
        {'A': 0, 'C': 2, 'G': 3, 'T': 5}
        
        >>> cal_count('banana')
        {'a': 0, 'b': 3, 'n': 4}
    """
    
    # Count how many times each letter appears in text, skipping $
    tally = Counter(string)
    if "$" in tally:
        del tally["$"]

    # Set a running total to 0
    running_total = 0

    # Make an empty result table
    result = {}

    # Walk through the letters alphabetically. Each letter's count is the total of all letters before it, which is also where its block
    # starts in the sorted first column.
    for letter in sorted(tally):
        result[letter] = running_total
        running_total = running_total + tally[letter]

    return result

In [28]:
#Test cal_count()

print(cal_count('ATGACG'))

print(cal_count('banana'))

# FOR each test case (input, expected answer):
test_cases = [
    ("ATGACG", {'A': 0, 'C': 2, 'G': 3, 'T': 5}),
    ("banana", {'a': 0, 'b': 3, 'n': 4}),
]

for input_text, expected in test_cases:
    result = cal_count(input_text)
    print(input_text, result, expected, result == expected)

{'A': 0, 'C': 2, 'G': 3, 'T': 5}
{'a': 0, 'b': 3, 'n': 4}
ATGACG {'A': 0, 'C': 2, 'G': 3, 'T': 5} {'A': 0, 'C': 2, 'G': 3, 'T': 5} True
banana {'a': 0, 'b': 3, 'n': 4} {'a': 0, 'b': 3, 'n': 4} True


In [29]:
def cal_occur(bwt_string: str) -> dict[str, list[int]]:
    """Function to calculate occurrences of each character up to each position.
    
    For each character and each position i, calculates how many times the
    character appears in the substring bwt_string[0:i].
    
    Args:
        bwt_string: The BWT string to analyze.
    
    Returns:
        A dictionary mapping each character to a list of occurrence counts,
        where occur[char][i] is the number of occurrences of char in
        bwt_string[0:i].
    
    Examples:
        >>> cal_occur('AG$CG')
        {'$': [0, 0, 1, 1, 1], 'A': [1, 1, 1, 1, 1], 'C': [0, 0, 0, 1, 1], 'G': [0, 1, 1, 1, 2]}
        
        >>> cal_occur('annb$aa')
        {'$': [0, 0, 0, 0, 1, 1, 1], 'a': [0, 1, 1, 1, 1, 2, 3], 'b': [0, 0, 0, 1, 1, 1, 1], 'n': [0, 0, 2, 2, 2, 2, 2]}
    """
    
    # Get each distinct character once, in alphabetical order
    letters = sorted(set(bwt_string))

    # Start every character with a count of 0 and an empty list
    counter = {}
    table = {}
    for letter in letters:
        counter[letter] = 0
        table[letter] = []

    # Read the string one position at a time. After counting the character at this position, record every character's running
    # total, so each list gets one entry per position.
    for i in range(len(bwt_string)):
        current_letter = bwt_string[i]
        counter[current_letter] = counter[current_letter] + 1

        for letter in letters:
            table[letter].append(counter[letter])
    
    return table

In [30]:
#Test cell for cal_occur

print(cal_occur('AG$CG'))
print(cal_occur('annb$aa'))


{'$': [0, 0, 1, 1, 1], 'A': [1, 1, 1, 1, 1], 'C': [0, 0, 0, 1, 1], 'G': [0, 1, 1, 1, 2]}
{'$': [0, 0, 0, 0, 1, 1, 1], 'a': [1, 1, 1, 1, 1, 2, 3], 'b': [0, 0, 0, 1, 1, 1, 1], 'n': [0, 1, 2, 2, 2, 2, 2]}


In [31]:
def update_range(
  lower: int, 
  upper: int, 
  count: dict[str, int], 
  occur: dict[str, list[int]], 
  a: str) -> tuple[int, int]:
    """Function to update range given character a.
    
    Updates the search range during backward search in the BWT pattern matching
    algorithm when processing character a.
    
    Args:
        lower: The lower boundary of the current range.
        upper: The upper boundary of the current range.
        count: Dictionary mapping each character to the count of lexicographically
            smaller characters.
        occur: Dictionary mapping each character to its occurrence counts at each
            position.
        a: The character being processed in the pattern.
    
    Returns:
        A tuple containing the updated lower and upper boundaries of the range.
    
    Note:
        This function assumes occur[a][-1] = 0 for boundary conditions.
    """

    #IF lower is 0:
        #letters above the block = 0
    if lower == 0:
        letters_above = 0
        
    #ELSE:
        #letters above the block = occur[the letter][lower - 1]

    else:
        letters_above = occur[a][lower - 1]


    #new lower = count[the letter] + letters above the block + 1
    #new upper = count[the letter] + occur[the letter][upper]

    new_lower = count[a] + letters_above + 1
    new_upper = count[a] + occur[a][upper]

    #return new lower, new upper

    return new_lower, new_upper

In [32]:
#Test update_range()

#count = cal_count of bananan
#occur = cal_occur of annb$aa
#FOR each test case (lower, upper, letter, expected answer):
    #run update_range
    #print the inputs, the result, and the expected answer

count = cal_count('banana')
occur = cal_occur('annb$aa')


test_cases = [
    (0, 6, 'a', (1, 3)),   # all rows -> rows starting with "a"
    (1, 3, 'n', (5, 6)),   # "a" rows -> "na" rows
    (5, 6, 'a', (2, 3)),   # "na" rows -> "ana" rows
    (1, 3, 'b', (4, 4)),   # "a" rows -> "ba" row
    (5, 6, 'n', (7, 6)),   # "na" rows -> no "nna" rows, so empty range
]

for lower, upper, letter, expected in test_cases:
    result = update_range(lower, upper, count, occur, letter)
    print(lower, upper, letter, result, expected, result == expected)

0 6 a (1, 3) (1, 3) True
1 3 n (5, 6) (5, 6) True
5 6 a (2, 3) (2, 3) True
1 3 b (4, 4) (4, 4) True
5 6 n (7, 6) (7, 6) True


In [33]:
def find_match(query: str, reference: str) -> list[int]:
    """Function to find exact matching by applying Burrows-Wheeler Transform.
    
    Searches for all occurrences of the query string within the reference string
    using the Burrows-Wheeler Transform algorithm for efficient pattern matching.
    
    Args:
        query: The pattern string to search for.
        reference: The text string to search within.
    
    Returns:
        A list of integers representing the 0-based starting positions of all
        occurrences of the query string within the reference string. Returns an
        empty list if no matches are found.
    
    Examples:
        >>> find_match('ana', 'banana')
        [1, 3]
        
        >>> find_match('xyz', 'banana')
        []
    """
    #IF query is empty:
        #return an empty list
    
    if not query:
        return []
    
    #Build Everything From The Reference

    #IF reference does not end with $:
        #add $ to end of reference

    if not reference.endswith('$'):
        reference = reference + '$'

    suffix_positions = suffix_array(reference)
    bwt_string = BWT_from_suffix_array(reference, suffix_positions)
    count = cal_count(reference)
    occur = cal_occur(bwt_string)

    #Search
    lower = 0
    upper = len(bwt_string) - 1

    #for each letter in query, from right to left:
    for letter in reversed(query):
        #IF the letter is not in count:
            #return an empty list
        if letter not in count:
            return []
        lower, upper = update_range(lower, upper, count, occur, letter)
        #lower, upper = update_range(lower, upper, count, occur, the letter)
        #IF lower > upper:
            #return an empty list
        if lower > upper:
            return []

        

    #Turn Rows Into Positions
    #make an empty result list
    result = []

    #FOR each row from lower to upper:
        #add suffix_positions[row] to the result
    #sort the result from smallest to largest

    for row in range(lower, upper + 1):
        result.append(suffix_positions[row])
    result.sort()

    return result

    #return result

In [34]:
#Test cell for find_match()

#FOR each test case (query, reference, expected answer):
    #run find_match
    #print the inputs, the result, and the expected answer

test_cases = [
    ("banana", "banana", [0]),        # occurs once
    ("a", "banana", [1, 3, 5]),       # occurs several times
    ("ana", "banana", [1, 3]),        # overlaps itself
    ("xyz", "banana", []),            # letters not in reference
    ("nab", "banana", []),            # letters exist, pattern doesn't
    ("go", "googol", [0, 3]),
]

for query, reference, expected in test_cases:
    result = find_match(query, reference)
    print(query, reference, result, expected, result == expected)

banana banana [0] [0] True
a banana [1, 3, 5] [1, 3, 5] True
ana banana [1, 3] [1, 3] True
xyz banana [] [] True
nab banana [] [] True
go googol [0, 3] [0, 3] True


In [35]:
def run_length_encode(bwt_string: str) -> str:
    """Run-length encode a BWT string as a compact character-count string.

    Scans the input BWT string from left to right and groups consecutive
    identical characters into runs, returning an encoded string where each
    run is represented as the character followed by its count.

    Args:
    bwt_string: The Burrows-Wheeler transformed string to encode.

    Returns:
    A run-length encoded string of the form 'a3n2b1$1a2', where each
    character is followed by the length of its consecutive run.

    Examples:
    >>> run_length_encode('aaabbc')
    'a3b2c1'

    >>> run_length_encode('annb$aa')
    'a1n2b1$1a2'
    """
    #IF bwt_string is empty:
        #return an empty string

    if bwt_string == "":
        return ""
    
    #make an empty result
    result = ""
    #current letter =  the first letter of bwt_string
    #run length = 1

    current_letter = bwt_string[0]
    run_length = 1

    #FOR each letter after the first one:
        #IF it is the same as the current letter:
            #add 1 to the run length
        #ELSE:
            #add the current letter and its run length to the result
            #current letter = this letter
            #run length = 1

    for letter in bwt_string[1:]:
        if letter == current_letter:
            run_length = run_length + 1
        else:
            result = result + current_letter + str(run_length)
            current_letter = letter
            run_length = 1
        
    #add the current letter and its run length to the result
    #return result

    result = result + current_letter + str(run_length)
    return result

In [36]:
#Test cell for run_length_encode()

#FOR each test case (input, expected answer):
    #run run_length_encode on the input
    #print the input, the result, and the expected answer

for input_text, expected in [
    ("aaabbc", "a3b2c1"),
    ("annb$aa", "a1n2b1$1a2"),
    ("", ""),
]:
    result = run_length_encode(input_text)
    print(input_text, result, expected, result == expected)

aaabbc a3b2c1 a3b2c1 True
annb$aa a1n2b1$1a2 a1n2b1$1a2 True
   True


In [37]:
def run_length_decode(encoded: str) -> str:
    """Decode a run-length encoded character-count string.

    Reconstructs the original string by parsing an encoded representation
    where each run is stored as a character followed by its count, and
    expanding each run back into repeated characters.

    Args:
    encoded: A run-length encoded string of the form 'a3n2b1$1a2'.

    Returns:
    The decoded string obtained by expanding all runs in the encoded input.

    Examples:
    >>> run_length_decode('a3b2c1')
    'aaabbc'

    >>> run_length_decode('a1n2b1$1a2')
    'annb$aa'
    """
    #make an empty result
    #start reading at position 0

    result = ""
    i = 0 

    #while there is still unread text:
        #read one character: this is the letter
        #move forward one position

    while i < len(encoded):
        letter = encoded[i]
        i += 1

        #make an empty holder for the digits
        #WHILE the current character is a digit:
            #add it to the digit holder
            #move forward one position
        digit_holder = ""
        while i < len(encoded) and encoded[i].isdigit():
            digit_holder += encoded[i]
            i += 1

        #turn the digit holder into a number
        #add the letter to the result that many times

        result += letter * int(digit_holder)

    return result

    #return result


In [38]:
#Test cell for run_length_decode()

#FOR each test case (input, expected answer)
    #run run_length_decode on the input
    #print the input, the result, and expected answer

test_cases = [
    ("aaabbc", "a3b2c1"),
    ("annb$aa", "a1n2b1$1a2"),
    ("aaaaaaaaaaaa", "a12"),    # run longer than 9
]

for text, expected in test_cases:
    encoded = run_length_encode(text)
    decoded = run_length_decode(encoded)
    print(text, encoded, expected, encoded == expected, "round trip:", decoded == text)

aaabbc a3b2c1 a3b2c1 True round trip: True
annb$aa a1n2b1$1a2 a1n2b1$1a2 True round trip: True
aaaaaaaaaaaa a12 a12 True round trip: True
